In [1]:
# -*- coding: utf-8 -*-
"""
Hierarchical clustering over K=438 clusters using centroid or center vectors (cosine).
Inputs (produced earlier):
  /kaggle/working/latte_clusters/k438/
    - clusters.parquet       (cluster_id, cid, ...)
    - centroids.npy          (438, D)
    - center_vecs.npy        (438, D)

Outputs:
  /kaggle/working/latte_clusters/k438_hclust/
    - linkage.npy
    - dendrogram.png         (600 dpi)
    - tree.newick
    - labels_k{m}.parquet    (if cutting by number of clusters)
      or labels_d{thr}.parquet (if cutting by distance threshold)
"""

import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster, to_tree

# -----------------------------
# Configuration
# -----------------------------
BASE_DIR     = "/kaggle/input/save-clusters/clusters_438"
OUT_DIR      = "/kaggle/working/hclust"
VECTOR_TYPE  = "centroid"   # "centroid" or "center"
LINKAGE      = "average"    # "average", "complete", "ward"(X)  # ward는 유클리드만
CUT_MODE     = "k"          # "k" or "distance"
CUT_VALUE    = 64           # if CUT_MODE="k": number of clusters (e.g., 64)
                            # if CUT_MODE="distance": cosine distance threshold (e.g., 0.15)

# -----------------------------
# Load vectors & ids
# -----------------------------
os.makedirs(OUT_DIR, exist_ok=True)

dfc = pd.read_parquet(os.path.join(BASE_DIR, "clusters.parquet"),
                      columns=["cluster_id", "cid"])
cluster_ids = dfc.sort_values("cid")["cluster_id"].to_list()  # ensure order by cid

if VECTOR_TYPE == "centroid":
    X = np.load(os.path.join(BASE_DIR, "centroids.npy"))      # (438, D)
elif VECTOR_TYPE == "center":
    X = np.load(os.path.join(BASE_DIR, "center_vecs.npy"))    # (438, D)
else:
    raise ValueError("VECTOR_TYPE must be 'centroid' or 'center'")

# -----------------------------
# Cosine distance (row-normalize → pdist('cosine'))
# -----------------------------
def l2_normalize_rows(A: np.ndarray, eps: float = 1e-9) -> np.ndarray:
    n = np.linalg.norm(A, axis=1, keepdims=True)
    return A / np.clip(n, eps, None)

Xn = l2_normalize_rows(X.astype(np.float32))
# condensed distance vector (length = 438*437/2)
Y = pdist(Xn, metric="cosine")  # 1 - dot for L2-normalized rows

# -----------------------------
# Hierarchical clustering (agglomerative)
# -----------------------------
# average/complete 등에서 condensed distance를 직접 넣으면 metric 인자는 무시됨.
Z = linkage(Y, method=LINKAGE)
np.save(os.path.join(OUT_DIR, "linkage.npy"), Z)

# -----------------------------
# Optional: cut the tree → flat labels
# -----------------------------
if CUT_MODE == "k":
    m = int(CUT_VALUE)
    labels = fcluster(Z, t=m, criterion="maxclust")  # 1..m
    lab_name = f"labels_k{m}.parquet"
elif CUT_MODE == "distance":
    thr = float(CUT_VALUE)
    labels = fcluster(Z, t=thr, criterion="distance")  # 1..C
    lab_name = f"labels_d{str(thr).replace('.','p')}.parquet"
else:
    raise ValueError("CUT_MODE must be 'k' or 'distance'")

df_labels = pd.DataFrame({
    "cluster_id": cluster_ids,
    "hclust_label": labels.astype(np.int32),
})
df_labels.to_parquet(os.path.join(OUT_DIR, lab_name), index=False)

# -----------------------------
# Dendrogram (600 dpi)
# -----------------------------
plt.figure(figsize=(48, 24))
# 노드 라벨을 cluster_id로 하면 그림이 과拥; leaf_label_func로 간략화 가능
dendrogram(
    Z,
    labels=[cid for cid in cluster_ids],  # 길면 잘림에 유의
    leaf_rotation=90,
    leaf_font_size=6,
    color_threshold=None,
)
plt.title(f"Hierarchical clustering (method={LINKAGE}, metric=cosine, vec={VECTOR_TYPE})")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "dendrogram.png"), dpi=600)
plt.close()

# -----------------------------
# Export Newick
# -----------------------------
# scipy to_tree → 재귀적으로 newick 변환
from io import StringIO
def _newick(node, newick, parent_dist, leaf_names):
    if node.is_leaf():
        return f"{leaf_names[node.id]}:{parent_dist - node.dist}{newick}"
    else:
        left = _newick(node.get_left(), "", node.dist, leaf_names)
        right = _newick(node.get_right(), "", node.dist, leaf_names)
        return f"({left},{right}):{parent_dist - node.dist}{newick}"

root, _ = to_tree(Z, rd=True)
newick_str = _newick(root, ";", root.dist, cluster_ids)
with open(os.path.join(OUT_DIR, "tree.newick"), "w", encoding="utf-8") as f:
    f.write(newick_str)

# -----------------------------
# Write a compact meta.json
# -----------------------------
meta = {
    "k": 438,
    "vector_type": VECTOR_TYPE,
    "linkage": LINKAGE,
    "metric": "cosine (pdist on L2-normalized rows)",
    "cut_mode": CUT_MODE,
    "cut_value": CUT_VALUE,
    "inputs": {
        "clusters_parquet": os.path.join(BASE_DIR, "clusters.parquet"),
        "vectors": "centroids.npy" if VECTOR_TYPE == "centroid" else "center_vecs.npy",
    },
    "outputs": {
        "linkage": "linkage.npy",
        "dendrogram": "dendrogram.png",
        "newick": "tree.newick",
        "labels": lab_name,
    }
}
with open(os.path.join(OUT_DIR, "meta.json"), "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("[OK] HClust done.")
print(f"Saved under: {OUT_DIR}")


[OK] HClust done.
Saved under: /kaggle/working/hclust


In [2]:
# ---- make searchable node index (no-cut) ----
from scipy.cluster.hierarchy import to_tree
import pandas as pd
import numpy as np
import os

OUT_DIR_SEARCH = os.path.join(OUT_DIR, "index")
os.makedirs(OUT_DIR_SEARCH, exist_ok=True)

# Z: linkage (shape (n-1, 4)), Xn: (n, D) L2-normalized vectors used for HClust
n = Xn.shape[0]                 # leaves = 438
D = Xn.shape[1]
root, nodes = to_tree(Z, rd=True)  # nodes: list of ClusterNode, length 2n-1

# SciPy node.id: 0..n-1 for leaves (원본 벡터 인덱스와 동일), n..2n-2 for 내부노드
# 각 노드의 centroid를 cosine 공간에서 계산: 
# - leaf: Xn[id]
# - internal: 자식 두 개의 centroid를 leaf-count로 가중 평균 후 L2 normalize
centroids_all = np.zeros((2*n - 1, D), dtype=np.float32)
size = np.zeros((2*n - 1,), dtype=np.int32)
is_leaf = np.zeros((2*n - 1,), dtype=bool)
left_id = np.full((2*n - 1,), -1, dtype=np.int32)
right_id = np.full((2*n - 1,), -1, dtype=np.int32)
parent_id = np.full((2*n - 1,), -1, dtype=np.int32)
level = np.zeros((2*n - 1,), dtype=np.int32)

# 1) leaf 초기화
for i in range(n):
    centroids_all[i] = Xn[i]
    is_leaf[i] = True
    size[i] = 1

# 2) 내부 노드 처리 (Z의 순서대로 만들면 bottom-up)
for row_idx, (a, b, dist, cnt) in enumerate(Z):
    node_idx = n + row_idx
    a = int(a); b = int(b)
    left_id[node_idx] = a
    right_id[node_idx] = b
    parent_id[a] = node_idx
    parent_id[b] = node_idx

    # 크기(leaf 수)
    size[node_idx] = int(cnt)

    # 가중 평균 후 정규화
    wa = size[a]; wb = size[b]
    v = (wa * centroids_all[a] + wb * centroids_all[b]) / (wa + wb)
    nrm = np.linalg.norm(v)
    if nrm > 0:
        v = v / nrm
    centroids_all[node_idx] = v

# 3) 레벨(루트=0 → 아래로 +1)
#    parent_id를 이용해 BFS로 계산
from collections import deque
root_id = n + Z.shape[0] - 1
q = deque([(root_id, 0)])
visited = set([root_id])
while q:
    nid, lv = q.popleft()
    level[nid] = lv
    for cid in (left_id[nid], right_id[nid]):
        if cid >= 0 and cid not in visited:
            visited.add(cid)
            q.append((cid, lv + 1))

# 4) leaves ↔ 원래 cluster_id 연결(리프 id 0..n-1은 cluster_ids와 정렬 맞춤)
leaf_index = np.full((2*n - 1,), -1, dtype=np.int32)
cluster_id_col = np.array([""] * (2*n - 1), dtype=object)
for i in range(n):
    leaf_index[i] = i
    cluster_id_col[i] = cluster_ids[i]

# 5) 저장
np.save(os.path.join(OUT_DIR_SEARCH, "centroids_all.npy"), centroids_all)  # (2n-1, D)

import pyarrow as pa, pyarrow.parquet as pq  # pandas만으로도 가능; 여기선 pandas로 저장
nodes_df = pd.DataFrame({
    "node_id": np.arange(2*n - 1, dtype=np.int32),
    "parent_id": parent_id.astype(np.int32),
    "left_id": left_id.astype(np.int32),
    "right_id": right_id.astype(np.int32),
    "level": level.astype(np.int32),
    "is_leaf": is_leaf,
    "size": size.astype(np.int32),
    "leaf_index": leaf_index,     # 리프일 때 0..n-1, 내부노드는 -1
    "cluster_id": cluster_id_col, # 리프면 "K0438-Cxxxx", 내부노드는 ""
    "centroid_path": ["centroids_all.npy"] * (2*n - 1),
})
nodes_df.to_parquet(os.path.join(OUT_DIR_SEARCH, "nodes.parquet"), index=False)

# (선택) 리프 멤버십은 기존 members.parquet를 그대로 사용:
# leaf_index == cid 와 매핑되므로, 특정 leaf(node_id=i) → cid=i → 해당 클러스터 멤버 전부 로드 가능
print("[OK] no-cut node index saved:",
      os.path.join(OUT_DIR_SEARCH, "nodes.parquet"),
      os.path.join(OUT_DIR_SEARCH, "centroids_all.npy"))


[OK] no-cut node index saved: /kaggle/working/hclust/index/nodes.parquet /kaggle/working/hclust/index/centroids_all.npy
